<a href="https://colab.research.google.com/github/YujiSue/Lecture/blob/master/%E3%83%81%E3%83%A5%E3%83%BC%E3%83%88%E3%83%AA%E3%82%A2%E3%83%AB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 医学部といえば...
### 昭和生まれの教員としては、やっぱり**手塚治虫のブラックジャック**が思い浮かびます。
### というわけで（？）ブラックジャックというゲームを知っているでしょうか？  
### トランプを使った非常にシンプルなカードゲームの１種です。詳しいルールは[こちら](https://casinocafe.jp/casinovenus/game/blackjack/)などを参照してください。
### 超簡単に言うと、カードを引いていって合計数が21に近い人が勝ち！というゲームです。
### このゲームを作る過程を利用してプログラミングを紹介していきましょう。  

### これから出てくる説明を<font color="red">上から順に正しく読んで実行すれば</font>、 **最後のゲームがプレイ** できます。
### もしプレイできない場合は、<font color="blue">途中飛ばしている箇所があると思われる</font>ので**きちんと見直し**ましょう。


# オブジェクトの準備

## まずはトランプのカードを作成します
### ブラックジャックはトランプを使ったカードゲームなので、何をおいてもまずトランプが必要です。

<img alt="このトランプ違う" src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/e5/Donald_Trump_Laconia_Rally%2C_Laconia%2C_NH_4_by_Michael_Vadon_July_16_2015_19.jpg/1280px-Donald_Trump_Laconia_Rally%2C_Laconia%2C_NH_4_by_Michael_Vadon_July_16_2015_19.jpg" width="300px"><font color="dimgray">このトランプ違う</font><br/>出典：wikipedia
<br>
<br>
### トランプはマークが♥︎♦︎♣️♠︎の４種類、それぞれに１〜１３の数字があります（今回ジョーカーは考えません）。
### 計52枚のカードが必要ですが、バカ正直に52枚分のプログラムを作るのも面倒です。
### まずは<font color="royalblue">【トランプカード (Card)】</font>という<font color="crimson">オブジェクト</font>を作って、この<font color="royalblue">【トランプカード】</font>に<font color="crimson">変数</font>として<font color="darkgreen">《マーク (suit)》</font>と<font color="darkgreen">《数 (number)》</font>を設定します。

In [ ]:
#@title トランプカードオブジェクトの構築

#@markdown まずはこのセルを実行してください。

#トランプカードのマークをリストアップ
suits = ['♥', '♦', '♠︎', '♣︎']

# トランプカードのオブジェクトを作る
class Card:
  ## 初期設定
  def __init__(self, index): #カードの番号(index)に応じてマークと数値を設定
    self.idx = index
    self.suit = suits[index // 13] #13で割った商をマークに対応
    self.number = index%13 + 1 #13で割った余りから数値を決定

  # カード情報を表示する機能を追加
  def printCard(self):
    print(self.suit, self.number)

In [ ]:
#@title トランプカードのテスト
#@markdown 上のコードを実行すると0から51番のどれかを指定すれば対応するカードが生成されます。
#@markdown ハートのエースが0番、最後の51番はクラブのキングになるはずなので確認しましょう。

heart_ace = Card(0)
heart_ace.printCard()

club_king = Card(51)
club_king.printCard()

## 次に山札（デッキ）を作ります
### ブラックジャックでは52枚組のトランプを４セット使って山札を作ります。
### この<font color="royalblue">【山札 (Deck)】</font>の<font color="crimson">オブジェクト</font>も作ってしまいましょう。
### <font color="royalblue">【山札】</font>は計208枚のカードオブジェクトからなる<font color="darkgreen">《カードの束 (cardset)》</font>を持っている、と考えてオブジェクトを作ります。
### ついでに、ゲーム用にカードのシャッフル関数<font color="purple">〈shuffle〉</font>もつけておきましょう。
### さらにさらに、ゲーム中の演算用に山札が何枚残っているかカウントする関数<font color="purple">〈remaining〉</font>もつけてしまいましょう。


In [ ]:
#@title 山札オブジェクトの構築

import random # ランダムに○○するためのモジュール

#トランプカードを52枚×４セット使って山札を作る
class Deck:
  def __init__(self):
    self.cardset = []
    for s in range(4):
      for index in range(52):
        self.cardset.append(Card(index))

  #シャッフル機能をつける
  def shuffle(self):
    shuffled_cards = []
    while len(self.cardset) > 0:
      #ランダムにカードを抜き出して新しい束にする
      card = self.cardset.pop(random.randrange(0, len(self.cardset)))
      shuffled_cards.append(card)
    self.cardset = shuffled_cards

  #残り枚数をカウント
  def remaining(self):
    return len(self.cardset)


In [ ]:
#@title 山札のテスト

# 新しい山札を用意
new_deck = Deck()

# 上から５枚を表示
for index in range(5):
  new_deck.cardset[index].printCard()

# シャッフル
new_deck.shuffle()

# シャッフル後
print('>> シャッフル後')

# 上から５枚を表示
for index in range(5):
  new_deck.cardset[index].printCard()

## 続けてプレイヤーを作ります

### ブラックジャックはチップ（財産）を賭けたプレイヤー同士の勝負です。
### この<font color="royalblue">【プレイヤー (Player)】</font>も１種の<font color="crimson">オブジェクト</font>になります。
### <font color="royalblue">【プレイヤー】</font>は変数として<font color="darkgreen">《保有チップ (asset)》《手札 (hand)》《タイプ (type)》《決断 (decision)》</font>を用意しましょう。
### また、山札からカードを引く関数<font color="purple">〈draw〉</font>、ゲーム終了時のリセット関数<font color="purple">〈reset〉</font>を用意します。


In [ ]:
#@title プレイヤーオブジェクトの構築
class Player:
  def __init__(self, chip): # 初期チップ量を指定
    self.asset = chip # 保有チップ
    self.hand = [] # 手札
    self.type = 'ユーザ' # プレイヤータイプとしてユーザ（あなた）をデフォルト設定
    self.decision = '' # 次の判断

  # 山札からカードを引く
  def draw(self, deck, count):
    for n in range(count):
      self.hand.append(deck.cardset.pop(0))

  # カード表示用
  def show(self, display_all = True):
    nodes = ''
    for index,card in enumerate(self.hand):
      attr = ''
      if (not display_all) and 0 < index:
        attr = 'back'
      else:
         if card.suit == '♥' or card.suit == '♦':
          attr = 'red'
      num = ''
      if card.number == 1:
        num = 'A'
      elif card.number == 11:
        num = 'J'
      elif card.number == 12:
        num = 'Q'
      elif card.number == 13:
        num = 'K'
      else:
        num = str(card.number)
      nodes += f'<div class="card {attr}">{card.suit}<br>{num}</div>'
    return nodes

  # 手札をリセット
  def reset(self):
    self.hand = []

## 最後にプレイヤーの亜種としてのディーラーも作ります。

### ディーラーは特殊ルールで動くプレイヤーです。
### <font color="royalblue">【プレイヤー (Player)】</font><font color="crimson">オブジェクト</font>を<font color="darkblue">継承</font>した<font color="royalblue">【ディーラー (dealer)】</font>を作ります。
### <font color="darkblue">継承≒プレイヤーとしての変数や関数は残したまま、ディーラー固有の機能を追加</font>
### <font color="royalblue">【ディーラー (dealer)】</font>は、引いた手札に応じて、次に打てる一手が単純な計算式で決定づけられています。
### この計算式を関数<font color="purple">〈next〉</font>としてディーラーにセットします。

In [ ]:
#@title 特殊ルールで動くディーラーの構築

class Dealer(Player):
  def __init__(self, chip):
    super().__init__(chip) # プレイヤーとしての初期化
    self.type = 'dealer' # タイプをディーラーとして上書き

  # 次にどうするかを決める
  def consider(self):
    ## 手札の合計が16以下の場合はヒット、17以上の場合はスタンド
    if bjsum(self.hand) <= 16:
      self.decision = 'ヒット'
    else:
      self.decision = 'スタンド'


# ゲームに関わる関数を用意

### オブジェクトを作り終わったら、いよいよゲームを進めるための仕組み作りに入ります。

### まずはゲームで繰り返し必要になる計算を関数として用意しましょう！

### 最初に書いた通り、ブラックジャックは手札の合計数を競うので、途中で何度も手札の合計数を求めることになります。

### 『合計』なら、ただ足し算すれば良いのでは？と思いきや、計算方法が特殊なので、まずはそのための計算関数を作りましょう。

In [ ]:
#@title 手札の合計算出法

def bjsum(cards):
  sum = 0 #合計値
  ace_num = 0 #エースの枚数
  for card in cards:
    # エースは扱いが特殊なのでとりあえず枚数だけ数える
    if card.number == 1:
      ace_num += 1
    # 10以下の場合は合計値に追加
    elif card.number <= 10:
      sum += card.number
    # 絵札は10扱い
    else:
      sum += 10
  if ace_num > 0:
    while ace_num > 0:
      # 21を越えない範囲であれば、エースは11として扱う
      if sum + ace_num - 1 <= 10:
        sum += 11
      # 21を超えそうなら1として扱う
      else:
        sum += 1
      ace_num -= 1
  return sum

### 続けて勝敗判定を考えます

### ブラックジャックでは賭け金がなくなったり、山札がなくなるまで、何度も勝ったり負けたりを繰り返します。

### この勝敗判定をおこなう関数も用意しましょう。

In [ ]:
#@title 勝敗判定方法

def judge(player1, player2): # player2がディーラーの想定
  #
  sum1 = bjsum(player1.hand)
  sum2 = bjsum(player2.hand)

  if sum1 > 21: # プレイヤーのバースト
    return {
        'message': '残念！プレイヤーのバースト！',
        'rate': 0
    }
  elif sum2 > 21: # ディーラーのバースト
    return {
        'message': 'やったね！ディーラーがバースト！',
        'rate': 2
    }
  elif sum1 < sum2: # ディーラーの方が21に近いのでプレイヤーの負け
    return {
        'message': '残念！ディーラーの勝ち',
        'rate': 0
    }
  elif sum1 > sum2: # プレイヤーの方が21に近いのでプレイヤーの勝ち
    return {
        'message': 'やったね！プレイヤーの勝ち',
        'rate': 2
    }
  else: ## 同じ得点の場合
    ### ブラックジャックが成立した場合
    if sum1 == 21:
      if len(player2.hand) == 2: # ディーラーが初手でブラックジャックなのでプレイヤーの負け
        return {
          'message': '残念！ディーラーの勝ち',
          'rate': 0
        }
      elif len(player1.hand) == 2: # プレイヤーが初手でブラックジャックなのでプレイヤーの勝ち
        return {
          'message': 'やったね！プレイヤーの勝ち',
          'rate': 2.5
        }
    return {
      'message': '引き分け',
      'rate': 1
    }


### あとはゲーム画面を用意しましょう

### Pythonだけだと、文字の表示は出来るけど、トランプっぽい絵の表示にはひと手間が必要です、

### ここでは、ユーザの選択を入力する所まで含めて、ウェブサイトの仕組みを流用していきます。

In [ ]:
#@title ゲーム画面をWebサイト方式で表示

import google.colab
from IPython.display import HTML

html = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <style>
        /* 全体のレイアウト */
        #bj-table {
            font-family: 'Helvetica Neue', Arial, sans-serif;
            background-color: #1e6b3e; /* カジノの緑テーブル */
            color: white;
            padding: 20px;
            border-radius: 12px;
            width: 450px;
            box-shadow: 0 8px 16px rgba(0,0,0,0.3);
            margin: 10px auto;
        }

        /* エリアごとの設定 */
        .area {
            background: rgba(0, 0, 0, 0.2);
            border-radius: 8px;
            padding: 15px;
            margin-bottom: 15px;
            min-height: 100px;
        }
        .area-title {
            font-size: 18px;
            text-transform: uppercase;
            letter-spacing: 1px;
            color: #bdffbd;
            margin-bottom: 10px;
        }

        /* 手札とカードの見た目 */
        .hand {
            display: flex;
            gap: 10px;
            min-height: 75px;
            align-items: center;
        }
        .card {
            background: white;
            color: black;
            width: 50px;
            height: 75px;
            border-radius: 5px;
            display: flex;
            flex-direction: column;
            justify-content: space-between;
            padding: 5px;
            font-weight: bold;
            font-size: 16px;
            box-shadow: 2px 2px 5px rgba(0,0,0,0.5);
            box-sizing: border-box;
        }
        .card.red { color: #d12e2e; } /* ハート・ダイヤ用 */
        .card.back {
            background: #d12e2e;
            border: 2px solid white;
            color: #d12e2e;
            justify-content: center;
            align-items: center;
        }

        /* メッセージと操作ボタン */
        #message-box {
            text-align: center;
            font-size: 18px;
            font-weight: bold;
            margin: 15px 0;
            min-height: 25px;
            color: #fffb00;
        }
        .controls {
            display: flex;
            justify-content: center;
            gap: 15px;
        }
        button {
            background: #ffffff;
            color: #1e6b3e;
            border: none;
            padding: 10px 20px;
            font-size: 16px;
            font-weight: bold;
            border-radius: 6px;
            cursor: pointer;
            transition: all 0.2s;
            box-shadow: 0 4px 6px rgba(0,0,0,0.2);
        }
        button:hover { background: #e0e0e0; transform: translateY(-1px); }
        button:disabled { background: #777; color: #aaa; cursor: not-allowed; transform: none; box-shadow: none; }
    </style>
</head>
<body>

<div id="bj-table">
    <!-- ディーラー（相手）エリア -->
    <div class="area">
        <div class="area-title">ディーラー</div>
        <div class="hand" id="dealer-hand">
        </div>
    </div>
    <div id="deck-pane" class="area-title">
        山札（<%deck-cards%>枚）
    </div>
    <!-- プレイヤー（自分）エリア -->
    <div class="area">
        <div id="your-info" class="area-title">プレイヤー（チップ：<%chip%>）</div>
        <div class="hand" id="player-hand">
        </div>
    </div>

    <!-- 状態アナウンス -->
    <div id="message-box"></div>

    <!-- 操作ボタン -->
    <div id="btn-pane" class="controls">
        <button id="btn-hit">ヒット</button>
        <button id="btn-stand">スタンド</button>
        <button id="btn-reset" style="display:none; background:#ffeb3b;">Next Game</button>
    </div>
</div>
</body>
</html>
"""

def showMain(deck, chip):
  code = html.replace('<%deck-cards%>', str(len(deck.cardset)))
  code = code.replace('<%chip%>', str(chip))
  display(HTML(code), metadata={'isolated': True})

def selectNext(player1, player2, deck, msg):
  ##
  return google.colab.output.eval_js(f"""
    (async function() {{
        const main = document.getElementById('bj-table');
        const deck = document.getElementById('deck-pane');
        deck.textContent = `山札（{len(deck.cardset)}枚）`;
        const info = document.getElementById('your-info');
        info.textContent = `プレイヤー（チップ：{player1.asset}）`;
        const msgbox = document.getElementById('message-box');
        msgbox.textContent = `{msg}`;
        const p1 = document.getElementById('player-hand');
        p1.innerHTML = `{player1.show(True)}`;
        const p2 = document.getElementById('dealer-hand');
        p2.innerHTML = `{player2.show(False)}`;
        const controler = document.getElementById('btn-pane');
        controler.style.display = 'flex';
        const res = await new Promise((resolve) => {{
          // イベントハンドラを定義
          const onClick = (e) => {{
            if (e.target.id === 'btn-hit') {{
              resolve('ヒット');
            }}
            else if (e.target.id === 'btn-stand') {{
              resolve('スタンド');
            }}
          }};
          // イベントリスナーを登録
          main.addEventListener('click', onClick);
        }});
        // Pythonへの戻り値
        return res;
    }})();""")

def updateUI(player1, player2, deck, msg, state=0):
  ##
  return google.colab.output.eval_js(f"""
    (function() {{
        const deck = document.getElementById('deck-pane');
        deck.textContent = `山札（{len(deck.cardset)}枚）`
        const info = document.getElementById('your-info');
        info.textContent = `プレイヤー（チップ：{player1.asset}）`;
        const controler = document.getElementById('btn-pane');
        controler.style.display = 'none';
        const msgbox = document.getElementById('message-box');
        msgbox.textContent = `{msg}`;
        const p1 = document.getElementById('player-hand');
        p1.innerHTML = `{player1.show(True)}`;
        const p2 = document.getElementById('dealer-hand');
        p2.innerHTML = `{player2.show(False if state==0 else True)}`;
    }})();""")

# ゲームプレイ


In [ ]:
#@markdown　ゲームの初期設定
最初のチップ = 10 #@param
賭け金 = 2 #@param

In [ ]:
#@title ゲームスタート

import time
from IPython.display import clear_output

#山札を用意
cards = Deck()

#シャッフル
cards.shuffle()

# プレイヤーの設定
chip = 最初のチップ
bet = 賭け金
you = Player(chip)
pc = Dealer(10000)

# ゲームの初期画面表示
showMain(cards, chip)

#ゲームスタート
#山札が無くなるか、プレイヤーのチップが無くなるまで繰り返す
while cards.remaining() > 0 and you.asset >= bet:
  #互いの手札をリセット
  you.reset()
  pc.reset()

  #チップをベット
  you.asset -= bet

  #まずはプレイヤーとディーラーが２枚ずつカードを引く
  you.draw(cards, 2)
  pc.draw(cards, 2)

  # プレイヤーがブラックジャックなら、もう何もする必要はない
  sum = bjsum(you.hand)
  go = sum != 21

  while go:
    # 手札の表示と次の手の選択
    you.decision = selectNext(you, pc, cards, "次の手を選んでください");

    # ヒットを選んだら、トランプの山からカードを１枚引く
    if you.decision == 'ヒット':
      you.draw(cards, 1)
      sum = bjsum(you.hand)
      # ブラックジャックかバーストしたらそこで終了
      if sum >= 21:
        go = False
    else:
      go = False

  # プレイヤーの状態を表示
  updateUI(you, pc, cards, "ディーラーのターン")

  # ディーラーのターン
  go = True
  while go:
    # ディーラーがカードを引いて表示
    pc.draw(cards, 1)
    time.sleep(1)
    updateUI(you, pc, cards, "ディーラーのターン")

    # ディーラーの次の一手
    pc.consider()
    if pc.decision == 'ヒット':
      pc.draw(cards, 1)
      time.sleep(1)
    elif pc.decision == 'スタンド':
      go = False

  #判定
  result = judge(you, pc)

  # 勝敗に応じて配当金をゲット
  result['payout'] = int(bet*result['rate'])
  you.asset += result['payout']
  you.decision = ''

  # 最後の状態を表示
  updateUI(you, pc, cards, result['message'], 1)
  time.sleep(3)

#最終的に手持ちのチップが増えれば勝利、減ったら負けです。
if you.asset > 最初のチップ:
  updateUI(you, pc, cards, 'あなたの勝利!!!', 1)
else:
  updateUI(you, pc, cards, 'ディーラーの勝利', 1)